# 00 - Setup and data cache

Run once per scan tree (Colab; the benchmark and the smoke test want a GPU). Writes `scan_config.json`, the cached splits and toys under `data/` and the cross sections. All logic is in `nsbi_scan.py`.

## Outputs

| path | content |
|---|---|
| `data/{sig,bkg,sbi}_{train,val}.npz` | kinematics, matrix elements and weights of the fixed 80/20 split (seed 42) |
| `data/summary.json`, `data/xs.json` | rows, weights and ESS per split; cross sections in fb |
| `data/toys_mu0p4.npz`, `data/toys_mu2p5.npz` | the 48 toys per mu |
| `scan_config.json` | the `ScanConfig` used by notebooks 01 to 03 |

Inputs are the MCFM CSVs in `spliced_21_columns/` and the toy CSVs in `observed_mu*/` next to the scan folder on Drive. The toys are weighted MC subsets without Poisson fluctuation, so the toy spread is MC noise, not coverage.

In [1]:
# --- Colab setup (run once per session) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !test -d /content/EveNet-Lite || git clone -q https://github.com/Dekiofsa/EveNet-Lite.git /content/EveNet-Lite
    !pip install -q --no-deps evenet-core==0.3.0
    !pip install -q --no-deps -e /content/EveNet-Lite
    !pip install -q lightning scipy scikit-learn wandb huggingface-hub pyyaml rich opt-einsum pyarrow tqdm regex
    sys.path.insert(0, "/content/EveNet-Lite")
def _find_scan_dir():
    """Locate the folder holding nsbi_scan.py on Drive (any folder name works)."""
    import glob as _glob
    cands = ["/content/drive/MyDrive/Research/NSBI/datascan",
             "/content/drive/MyDrive/Research/NSBI/Training Statistics Scan"]
    for pat in ("/content/drive/MyDrive/Research/NSBI/*/nsbi_scan.py",
                "/content/drive/MyDrive/*/nsbi_scan.py",
                "/content/drive/MyDrive/*/*/nsbi_scan.py"):
        if any(os.path.isfile(os.path.join(c, "nsbi_scan.py")) for c in cands):
            break
        cands += sorted({os.path.dirname(p) for p in _glob.glob(pat)})
    for c in cands:
        if os.path.isfile(os.path.join(c, "nsbi_scan.py")):
            return c
    raise FileNotFoundError("nsbi_scan.py not found on Drive: copy the whole scan folder "
                            "(nsbi_scan.py + notebooks) under MyDrive/Research/NSBI/ and rerun this cell")
SCAN_DIR = _find_scan_dir() if IN_COLAB else os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, SCAN_DIR)
import nsbi_scan as ns
print("nsbi_scan from", ns.__file__)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.8/115.8 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for evenet-lite (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 31.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
evenet-core 0.3.0 requires numpy==1.26.4, but you have numpy 2.1.3 which is incompatible.
nsbi_scan from /content/drive/MyDriv

In [2]:
# ============================ CONFIG (edit here) ============================
import dataclasses, json, time
import numpy as np
import pandas as pd

RAW_DATA_DIR = os.environ.get("NSBI_RAW_DATA_DIR", "/content/drive/MyDrive/Research/NSBI/spliced_21_columns")
TOYS_ROOT = os.environ.get("NSBI_TOYS_ROOT", "/content/drive/MyDrive/Research/NSBI")   # holds observed_mu0p4/, observed_mu2p5/
SCAN_ROOT = ns.resolve_root(SCAN_DIR)

cfg = ns.ScanConfig(
    root=SCAN_ROOT,
    raw_data_dir=RAW_DATA_DIR,
    toys_root=TOYS_ROOT,
    fractions=[0.01, 0.03, 0.10, 0.30, 1.00],
    seeds=[0, 1, 2],
    epoch_size=1_024_000,
    max_epochs=50,                               # cap = 100k optimizer steps
    early_stop_patience=5,
    num_workers=0,
    keep_checkpoint=False,                       # keep final.pt per run (80-240 MB each)
    wandb_project=None,
)
print(cfg)
print("config saved to", cfg.save())

ScanConfig(root='/content/drive/MyDrive/Research/NSBI/Training Statistics Scan', raw_data_dir='/content/drive/MyDrive/Research/NSBI/spliced_21_columns', toys_root='/content/drive/MyDrive/Research/NSBI', train_frac=0.8, split_seed=42, fractions=[0.01, 0.03, 0.1, 0.3, 1.0], variants=['scratch', 'pretrained', 'frozen'], seeds=[0, 1, 2], tasks=['sig_over_bkg', 'sbi_over_bkg'], batch_size=512, epoch_size=1024000, max_epochs=50, early_stop_patience=5, val_monitor_rows=262144, num_workers=0, scratch_lr=0.001, scratch_weight_decay=0.01, lumi=300.0, mu_true_values=[0.4, 2.5], mu_min=0.0, mu_max=4.0, mu_steps=401, calib_ref_events=1000000, keep_checkpoint=False, wandb_project=None, wandb_entity=None)
config saved to /content/drive/MyDrive/Research/NSBI/Training Statistics Scan/scan_config.json


In [ ]:
summary = ns.cache_splits(cfg)

counts = pd.DataFrame(
    [{"process": p, "part": part, **summary[f"{p}_{part}"]} for p in ns.PROCESSES for part in ("train", "val")]
)
counts["ess_over_rows"] = counts["ess"] / counts["rows"]
display(counts)
print("total rows per process:", {p: summary[f"{p}_total_rows"] for p in ns.PROCESSES})

In [ ]:
xs = ns.cache_cross_sections(cfg)
print(json.dumps(xs, indent=2))

closure = xs["sig"] + xs["int"] + xs["bkg"]
print(f"sig + int + bkg = {closure:.6f} fb   vs   sbi = {xs['sbi']:.6f} fb   (rel. diff {closure / xs['sbi'] - 1:+.2e})")
assert abs(closure - xs["sbi"]) <= 1e-3 * abs(xs["sbi"]), "SBI cross section does not close to sig+int+bkg"

yields = pd.DataFrame([{"mu": mu, "nu_expected": ns.expected_yield(xs, mu, cfg.lumi)} for mu in (0.4, 1.0, 2.5)])
print(f"expected event yields nu(mu) at {cfg.lumi:.0f} fb^-1:")
display(yields)

In [ ]:
toy_info = ns.cache_toys(cfg)
print(toy_info)

toys_df = ns.check_toys(cfg, xs)
per_tag = toys_df.groupby("tag").agg(
    n_toys=("toy", "nunique"),
    rows=("rows", "sum"),
    rows_per_toy=("rows", "mean"),
    nu_expected=("nu_expected", "first"),
    mean_sum_n=("sum_n", "mean"),
    max_abs_rel_dev=("rel_dev", lambda s: s.abs().max()),
).reset_index()
display(per_tag)

In [ ]:
display(ns.training_event_table(cfg))

wt = {}
for p in ns.PROCESSES:
    d = np.load(cfg.data_dir / f"{p}_train.npz")
    wt[p] = d["wt"]
    if p == "sbi":
        msq = d["msq"]
        i_sbi, i_bkg = ns.MSQ_COLS.index("msq_sbi_sm"), ns.MSQ_COLS.index("msq_bkg_sm")
        wt["sbi_as_bkg"] = wt["sbi"] * msq[:, i_bkg] / msq[:, i_sbi]
        del msq

rows = []
for f in cfg.fractions:
    for seed in cfg.seeds:
        row = {"fraction": f, "seed": seed}
        for name, w in wt.items():
            idx = ns.subsample_indices(len(w), f, ns.subsample_seed(seed))
            ws = w[idx]
            row[f"{name}_n"] = int(len(idx))
            row[f"{name}_ess"] = ns.effective_sample_size(ws)
            row[f"{name}_top1_share"] = float(ws.max() / ws.sum())
            row[f"{name}_top10_share"] = float(np.partition(ws, -10)[-10:].sum() / ws.sum()) if len(ws) > 10 else 1.0
        rows.append(row)
ess_df = pd.DataFrame(rows)
display(ess_df)

ess_cols = [c for c in ess_df.columns if c.endswith("_ess")]
print("ESS averaged over seeds (and its relative spread over seeds):")
ess_mean = ess_df.groupby("fraction")[ess_cols].mean().round(0)
ess_rel_std = (ess_df.groupby("fraction")[ess_cols].std() / ess_df.groupby("fraction")[ess_cols].mean()).round(3)
display(pd.concat({"mean": ess_mean, "rel_std": ess_rel_std}, axis=1))
print("largest single-event weight share, max over seeds (compare with the full-train row):")
display(ess_df.groupby("fraction")[[c for c in ess_df.columns if c.endswith("_top1_share")]].max())
del wt

In [ ]:
try:
    import torch
except ImportError as exc:
    raise RuntimeError("torch / EveNet-Lite are needed here: run this cell on Colab with a GPU runtime") from exc
from nsbi.data import FEATURE_NAMES, NORMALIZATION_RULES, make_classifier_data
from evenet_lite.callbacks import Callback

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())
else:
    print("WARNING: no GPU - the numbers below are not representative of the scan")

BENCH_EPOCH_SIZE = 102_400
BENCH_VAL_ROWS = 10_000

P = ns.task_processes(cfg, "sig_over_bkg", 0.01, 0)
train_data = make_classifier_data(P["num_train"], P["den_train"], denominator_reweight=P["reweight"])
idx_num = ns.subsample_indices(len(P["num_val"]), BENCH_VAL_ROWS / len(P["num_val"]), ns.VAL_MONITOR_SEED)
idx_den = ns.subsample_indices(len(P["den_val"]), BENCH_VAL_ROWS / len(P["den_val"]), ns.VAL_MONITOR_SEED)
val_data = make_classifier_data(P["num_val"].take(idx_num), P["den_val"].take(idx_den), denominator_reweight=P["reweight"])
print(f"train rows: {len(train_data[1])}  val rows: {len(val_data[1])}")

class StepTimer(Callback):
    """Wall time from the first to the last *training* step only: DataLoader worker start-up (train and
    val), the validation pass and the fit-level overheads are excluded.  on_batch_end also fires during
    validation, hence the trainer.model.training guard (same as EveNet-Lite's DebugCallback)."""

    def __init__(self):
        self.t_first = None
        self.t_last = None
        self.n = 0

    def on_batch_end(self, trainer, epoch, batch_idx, batch, loss, metrics=None):
        if not trainer.model.training:
            return
        now = time.time()
        if self.t_first is None:
            self.t_first = now
        self.t_last = now
        self.n += 1

    @property
    def train_seconds(self):
        return self.t_last - self.t_first if self.n > 1 else float("nan")

    @property
    def steps_per_s(self):
        return (self.n - 1) / self.train_seconds if self.n > 1 else float("nan")

bench_rows = []
for i, nw in enumerate((0, 2, 0, 2)):
    cfg_bench = dataclasses.replace(cfg, num_workers=nw)
    ns.set_seeds(0)
    clf = ns.build_variant("scratch", cfg_bench)
    timer = StepTimer()
    t0 = time.time()
    clf.fit(
        train_data=train_data,
        val_data=val_data,
        feature_names=FEATURE_NAMES,
        normalization_rules=NORMALIZATION_RULES,
        callbacks=[timer],
        epochs=1,
        epoch_size=BENCH_EPOCH_SIZE,
        batch_size=cfg.batch_size,
        sampler="weighted",
        checkpoint_path=None,
    )
    wall = time.time() - t0
    steps = int(clf.trainer.global_step)
    if timer.n != steps:
        print(f"WARNING: timer saw {timer.n} training steps but the trainer counted {steps}")
    bench_rows.append({
        "pass": i // 2, "num_workers": nw, "steps": steps, "train_seconds": timer.train_seconds,
        "steps_per_s": timer.steps_per_s, "fit_wall_seconds": wall,
        "device": str(clf.trainer.device), "n_trainable": ns.count_trainable(clf),
    })
    print(f"pass {i // 2} num_workers={nw}: {steps} steps, first->last step {timer.train_seconds:.1f} s "
          f"-> {timer.steps_per_s:.1f} steps/s (fit wall {wall:.1f} s incl. worker start-up, normalisation, val)")
    del clf
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

bench = pd.DataFrame(bench_rows)
display(bench)
rate = bench.groupby("num_workers").steps_per_s.max()     # best pass per worker setting
best_nw, best_rate = int(rate.idxmax()), float(rate.max())
steps_per_epoch = int(np.ceil(cfg.epoch_size / cfg.batch_size))
print(f"best: num_workers={best_nw} at {best_rate:.1f} steps/s (max over passes)")
print(f"-> one scan epoch ({cfg.epoch_size} rows = {steps_per_epoch} steps) ~ {steps_per_epoch / best_rate / 60:.1f} min of training,"
      f" {cfg.max_epochs}-epoch cap ~ {cfg.max_epochs * steps_per_epoch / best_rate / 3600:.1f} h per run (plus val monitoring and r(x) export)")
if best_nw != cfg.num_workers:
    print(f"NOTE: cfg.num_workers={cfg.num_workers} but num_workers={best_nw} was faster; consider editing the CONFIG cell and re-saving")
del train_data, val_data, P

In [ ]:
import logging, re, shutil, tempfile
try:
    import torch
except ImportError as exc:
    raise RuntimeError("torch / EveNet-Lite are needed here: run this cell on Colab with a GPU runtime") from exc

SMOKE_ROOT = "/content/smoke" if IN_COLAB else os.path.join(tempfile.gettempdir(), "nsbi_smoke")
SMOKE_FORCE = True

cfg_smoke = dataclasses.replace(
    cfg, root=SMOKE_ROOT, epoch_size=51_200, max_epochs=2, early_stop_patience=0,
    val_monitor_rows=50_000, calib_ref_events=50_000, wandb_project=None,
)
os.makedirs(cfg_smoke.root, exist_ok=True)
link = cfg_smoke.data_dir
if link.is_symlink():
    link.unlink()
elif link.exists():
    shutil.rmtree(link)
try:
    os.symlink(str(cfg.data_dir), str(link), target_is_directory=True)
except OSError as exc:
    print(f"symlink failed ({exc}); copying the data cache instead")
    shutil.copytree(cfg.data_dir, link)
assert (link / "summary.json").exists() and (link / "xs.json").exists(), f"{link} does not see the cached data"
cfg_smoke.save()
print("smoke root:", cfg_smoke.root, "| data ->", os.path.realpath(link))

class LogCapture(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.INFO)
        self.messages = []

    def emit(self, record):
        self.messages.append(record.getMessage())

def _fail_msg(variant, task):
    f = ns.run_dir(cfg_smoke, variant, task, 0.01, 0) / "failed.json"
    return json.load(open(f))["traceback"] if f.exists() else f"(no failed.json under {f.parent})"

t_smoke = time.time()
m_scratch = ns.train_one(cfg_smoke, "scratch", "sig_over_bkg", 0.01, 0, device="auto", force=SMOKE_FORCE)
assert m_scratch is not None, "scratch S/B run failed:\n" + _fail_msg("scratch", "sig_over_bkg")

capture = LogCapture()
logging.getLogger().addHandler(capture)
try:
    m_pre = ns.train_one(cfg_smoke, "pretrained", "sbi_over_bkg", 0.01, 0, device="auto", force=SMOKE_FORCE)
finally:
    logging.getLogger().removeHandler(capture)
assert m_pre is not None, "pretrained SBI/B run failed:\n" + _fail_msg("pretrained", "sbi_over_bkg")
hits = [m for m in (re.search(r"Loaded (\d+) / (\d+) layers", msg) for msg in capture.messages) if m]
assert hits, "no 'Loaded N / M layers' soft-load summary was logged during the pretrained build"
n_loaded, n_total = (int(g) for g in hits[-1].groups())
print(f"pretrained soft-load: {n_loaded} / {n_total} layers taken from the HF checkpoint")
assert n_loaded >= 400, f"only {n_loaded} layers loaded from the pretrained checkpoint (expected ~500 of ~535)"

m_pre_sig = ns.train_one(cfg_smoke, "pretrained", "sig_over_bkg", 0.01, 0, device="auto", force=SMOKE_FORCE)
assert m_pre_sig is not None, "pretrained S/B run failed:\n" + _fail_msg("pretrained", "sig_over_bkg")

clf_frozen = ns.build_variant("frozen", cfg_smoke)
n_head = ns.count_trainable(clf_frozen)
n_all = sum(p.numel() for p in clf_frozen.model.parameters())
print(f"frozen: {n_head:,} trainable of {n_all:,} parameters")
assert 0 < n_head < 0.2 * n_all, "frozen variant did not freeze the backbone"
del clf_frozen
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ---- what every run exports ----
runs = ns.collect_runs(cfg_smoke)
show_cols = [c for c in ["variant", "task", "frac", "seed", "n_num_train", "n_den_train", "ess_num_train", "ess_den_train",
                         "n_trainable_params", "epochs_run", "best_epoch", "best_step", "best_val_loss_monitor",
                         "val_bce_full", "val_auc_full", "C_ref", "C_sub", "C_val", "closure_chi2_ndf",
                         "fit_seconds", "total_seconds"] if c in runs.columns]
display(runs[show_cols])

meas = ns.measure_all(cfg_smoke, runs, diagonal_only=True)
assert len(meas), "measure_all found no (S/B, SBI/B) pair of the same variant and seed"
per_seed, over = ns.aggregate(meas)
display(over)

n_toys_expected = {tag: info["n_toys"] for tag, info in ns.cache_toys(cfg).items()}
ok = (
    len(over) == len(cfg.mu_true_values)
    and bool(np.isfinite(over["bias_mean"]).all())
    and all(int(per_seed.loc[per_seed.mu_true == mu, "n_toys"].iloc[0]) == n_toys_expected[ns.MU_TAGS[mu]] for mu in cfg.mu_true_values)
)
assert ok, "measurement output has an unexpected shape - inspect `over` and `per_seed`"
print(f"SMOKE TEST PASS: 3 runs trained, frozen build checked, r(x) exported, {len(meas)} toy measurements aggregated "
      f"({(time.time() - t_smoke) / 60:.1f} min). Untrained-model biases above are expected to be large.")

## Next

Notebook 01 trains the tiers (cheap fractions first, resumable), 02 measures mu on the toys, 03 makes the figures. Every point has the same budget: 2,000-step epochs, at most 50, early stopping with patience 5.